In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import  matplotlib.pyplot as plt

dataset = pd.read_csv("heart.csv")

sedanten_fitur = ['Age','Sex','ChestPainType','RestingBP','Cholesterol','FastingBS','RestingECG','MaxHR','ExerciseAngina','Oldpeak','ST_Slope']
fitur_continu = ['RestingBP', 'Cholesterol', 'FastingBS', 'HeartDisease']
fitur_diskrit = ['Age', 'MaxHR', 'Oldpeak',  'HeartDisease']
fitur_ordinal = ['ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope',  'HeartDisease']
fitur_nominal = ['Sex']
all_feature_cols = ['Age','Sex','ChestPainType','RestingBP','Cholesterol','FastingBS','RestingECG','MaxHR','ExerciseAngina','Oldpeak','ST_Slope']

map_chest = {'ASY' : 4, 'NAP' : 1, 'ATA' : 2, 'TA' : 3}
map_resting = {'Normal' : 1, 'LVH' : 2, 'ST' : 3}
map_exercise = {'N' : 1, 'Y' : 1}
map_slope = {'Flat' : 1, 'Up' : 3, 'Down' : 2}

def label_encode(dataset, col):
    for cols in col:
        dataset[cols] = pd.factorize(dataset[cols])[0]

top_6_cols = ['Oldpeak', 'ChestPainType', 'Age', 'RestingBP', 'Sex', 'RestingECG']
top_5_cols = ['Oldpeak', 'ChestPainType', 'Age', 'RestingBP', 'Sex']
top_3_cols = ['Age', 'ChestPainType', 'Oldpeak']

dataset= label_encode(dataset, fitur_nominal)
dataset['ChestPainType'] = dataset['ChestPainType'].map(map_chest)
dataset['RestingECG'] = dataset['RestingECG'].map(map_resting)
dataset['ExerciseAngina'] = dataset['ExerciseAngina'].map(map_exercise)
dataset['ST_Slope'] = dataset['ST_Slope'].map(map_slope)


def stratified_split(dataset, target_column, training_size=0.8, random_state=42, frac=1):
    np.random.seed(random_state)
    train_list, test_list = [], []

    for class_value in dataset[target_column].unique():
        class_data = dataset[dataset[target_column] == class_value]
        class_data = class_data.sample(random_state=random_state, frac=frac)

        split_idx = int(len(class_data) * training_size)
        train_list.append(class_data.iloc[:split_idx])
        test_list.append(class_data.iloc[split_idx:])

    train_set = pd.concat(train_list).reset_index(drop=True)
    test_set = pd.concat(test_list).reset_index(drop=True)
    return train_set, test_set

dataste_training, dataset_testing = stratified_split(dataset, "HeartDisease")
def random_oversampling(dataset, target_column, additional_size = None):

    classes = dataset[target_column].value_counts()
    resampled_data = [dataset]

    for label, size in classes.items:
        if additional_size and additional_size.get(label, 0):
            additional_data = dataset[dataset[label] == label]
            minority_class_data = additional_data.sample(n = additional_size[label], random_state = 42)
            resampled_data.append(minority_class_data)

    final_dataset = pd.concat(resampled_data).sample(random_state=42, replace=True).reset_index(drop=True)
    return final_dataset
X_train = dataste_training[top_6_cols]
X_test = dataset_testing[top_6_cols]

y_train = dataste_training["HeartDisease"]
y_test = dataset_testing["HeartDisease"]
def to_float(dataset, col):
    for cols in col:
        dataset[cols] = dataset[cols].astype(float)
    return dataset

dataset = to_float(dataset, fitur_continu)
dataset = to_float(dataset, fitur_diskrit)
def pca_fit(X, n_components):
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_scaled = (X - X_mean) / X_std

    cov = np.cov(X_scaled, rowvar=False)

    if np.ndim(cov) == 0:
        cov = np.array([[cov]])

    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    sorted_indices = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]

    components = eigenvectors[:, :n_components]
    X_pca = np.dot(X_scaled, components)
    explained_variance = eigenvalues[:n_components]
    return X_pca, components, X_mean, X_std, explained_variance

def pca_transform(X, components, X_mean, X_std):
    X_scaled = (X - X_mean) / X_std
    return np.dot(X_scaled, components)
def mca_fit(X_cat, n_components):
    X_dummies = pd.get_dummies(X_cat)
    X_matrix = X_dummies.to_numpy()

    row_sums = X_matrix.sum(axis=1, keepdims=True)
    col_sums = X_matrix.sum(axis=0, keepdims=True)
    total = X_matrix.sum()

    expected = row_sums @ col_sums / total
    Z = (X_matrix - expected) / np.sqrt(expected)

    U, S, VT = np.linalg.svd(Z, full_matrices=False)
    X_mca = U[:, :n_components] * S[:n_components]
    components = VT[:n_components, :]
    return X_mca, components, X_dummies.columns

def mca_transform(X_cat, components, dummy_columns):
    X_dummies = pd.get_dummies(X_cat)
    X_dummies = X_dummies.reindex(columns=dummy_columns, fill_value=0)

    X_matrix = X_dummies.to_numpy()
    row_sums = X_matrix.sum(axis=1, keepdims=True)
    col_sums = X_matrix.sum(axis=0, keepdims=True)
    total = X_matrix.sum()
    expected = row_sums @ col_sums / total
    Z = (X_matrix - expected) / np.sqrt(expected)

    X_proj = Z @ components.T
    return X_proj
def famd_fit(df, n_components):

    float_cols = df.select_dtypes(include = ['float']).columns
    int_cols = df.select_dtypes(include = ['int']).columns

    X_num = df[float_cols].to_numpy()
    X_pca, pca_components, pca_mean, pca_std, _ = pca_fit(X_num, X_num.shape[1])

    X_cat = df[int_cols]
    X_mca, mca_components, dummy_columns = mca_fit(X_cat, min(n_components, 50))

    X_combined = np.hstack([X_pca, X_mca])
    X_combined_mean = np.mean(X_combined, axis=0)
    X_combined_std = np.std(X_combined, axis=0)
    X_famd, famd_components, _, _, _ = pca_fit(X_combined, n_components)

    return {
        'float_cols' : float_cols,
        'int_cols' : int_cols,
        'X_num' : X_num,
        'X_pca' : X_pca,
        'pca_components' : pca_components,
        'pca_mean' : pca_mean,
        'pca_std' : pca_std,
        'X_mca' : X_mca,
        'mca_components' : mca_components,
        'dummy_columns' : dummy_columns,
        'X_famd' : X_famd,
        'famd_components' : famd_components,
        'X_combined_mean' : X_combined_mean,
        'X_combined_std' : X_combined_std
    }

def famd_transform(df, model):
    
    X_num = df[model['float_cols']].to_numpy()
    safe_std = np.where(model['pca_std'] == 0, 1, model['pca_std'])
    X_scaled = (X_num - model['pca_mean']) / safe_std
    X_pca = np.dot(X_scaled, model['pca_components'])

    X_cat = df[model['int_cols']]
    X_mca = mca_transform(X_cat, model['mca_components'], model['dummy_columns'])

    X_combined = np.hstack([X_pca, X_mca])
    safe_std_combined = np.where(model['X_combined_std'] == 0, 1, model['X_combined_std'])
    X_combined_scaled = (X_combined - model['X_combined_mean']) / safe_std_combined
    X_famd = np.dot(X_combined_scaled, model['famd_components'])

    return X_famd
famd_model = famd_fit(X_train, n_components= 2)
X_train_famd = famd_model['X_famd']
X_test_famd = famd_transform(X_test, famd_model)

famd_df = pd.DataFrame(X_train_famd, columns=['Dim 1', 'Dim 2'])
famd_df['target'] = y_train.values
class KNN:
    def __init__(self, k=5, p=2):
        self.k = k
        self.p = p

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def minkowski_distance(self, a, b):
        return np.sum(np.abs(a - b) ** self.p) ** (1 / self.p)
    
    def predict(self, X):
        X_test = X
        distances = np.zeros((X_test.shape[0], self.X_train.shape[0]))

        for i, test_point in enumerate(X_test):
            for j, train_point in enumerate(self.X_train):
                distances[i, j] = self.minkowski_distance(test_point, train_point)

        nearest_neighhbor_indices = np.argsort(distances, axis=1)[:, :self.k]
        neighbors = np.array(self.y_train)[nearest_neighhbor_indices]

        y_pred = []
        for neighbor_classes in neighbors:
            classes, counts = np.unique(neighbor_classes, return_counts=True)
            predicted_classes = classes[np.argmax(counts)]
            y_pred.append(predicted_classes)

        return np.array(y_pred)
    
    def plot_neighbors(self, input_point, y_train, zoom=False):
        distances = np.array([self.minkowski_distance(input_point, train_point) for train_point in self.X_train])
        nearest_neighbors_indices = np.argsort(distances)[:self.k]

        plt.figure(figsize=(12, 8))
        scatter = plt.scatter(self.X_train[:, 0], self.X_train[:, 1], c=y_train, cmap="coolwarm", alpha=0.7, label="Training Data")
        plt.colorbar(scatter, label="Classes")

        plt.scatter(input_point[0], input_point[1], c="yellow", edgecolors="black", s=200, marker="*", label="Input Data")

        if zoom:
            all_x = [input_point[0]] + [self.X_train[idx][0] for idx in nearest_neighbors_indices]
            all_y = [input_point[1]] + [self.X_train[idx][1] for idx in nearest_neighbors_indices]

            x_padding = (max(all_x) - min(all_x)) * 0.2
            y_padding = (max(all_y) - min(all_y)) * 0.2

            offset_y = y_padding * 0.15

            plt.xlim(min(all_x) - x_padding, max(all_x) + x_padding)
            plt.ylim(min(all_y) - y_padding, max(all_y) + y_padding)

            print("Nearest Neighbor Label:")
            for i, idx in enumerate(nearest_neighbors_indices):
                print(f"Nearest Neighbor {i+1}: Label - {y_train[idx]}")

        else:
            offset_y = 0.2

        cmap = plt.cm.coolwarm
        norm = plt.Normalize(vmin=0, vmax=0)

        for i, idx in enumerate(nearest_neighbors_indices):
            neighbor_point = self.X_train[idx]
            color = cmap(norm(y_train[idx]))
            label = "Nearest Neighbors" if i == 0 else None
            plt.scatter(neighbor_point[0], neighbor_point[1], facecolors=color, edgecolors="black", s=75, linewidths=2, label=label)
            plt.plot([input_point[0], neighbor_point[0]], [input_point[1], neighbor_point[1]], "black", linestyle="dashed", alpha=0.5)
            plt.text(neighbor_point[0], neighbor_point[1] + offset_y, f"{i+1}", fontsize=9, color="black", ha="center")

        plt.xlabel("Dimensi 1 FAMD")
        plt.ylabel("Dimensi 2 FAMD")
        plt.title("FAMD Scatter Plot dengan Input Data Baru")
        plt.legend()
        plt.show()
model1 = KNN(k=7, p=1)
model1.fit(X_train_famd, y_train)
y_pred_train1 = model1.predict(X_train_famd)
y_pred_test1 = model1.predict(X_test_famd)
model2 = KNN(k=7, p=2)
model2.fit(X_train_famd, y_train)
y_pred_train2 = model2.predict(X_train_famd)
y_pred_test2 = model2.predict(X_test_famd)
input_data = [2.7, 1, 33, 175, 0, 3]
input_df = pd.DataFrame([input_data], columns=top_6_cols)
input_df[['Age', "RestingBP"]] = input_df[['Age', "RestingBP"]].astype(float)
input_famd = famd_transform(input_df, famd_model)[0]
input_famd

TypeError: 'NoneType' object is not subscriptable